        # 0001 — PDF text extraction

        **Question.** Which PDF extractor gives page-accurate text for the
        French corpus, fast and fully offline?

        **Options.** `pymupdf` (PyMuPDF), `pypdf`.

        **Metric.** pages detected, chars/page, seconds/PDF. Page-number fidelity
        is structural: a list indexed by page preserves `#page=N`.

        **Feeds.** [ADR-0005](../../docs/adr/0005-page-aware-ingestion.md),
        `brh_core/ingestion/extract.py`.

        > Requires the corpus: run `make seed` from the repo root first.
        


In [ ]:
        import os
        from pathlib import Path


        def find_corpus() -> Path:
            if env := os.environ.get("BRH_RAW_DIR"):
                return Path(env)
            for base in (Path.cwd(), *Path.cwd().parents):
                if (base / "data" / "raw").is_dir():
                    return base / "data" / "raw"
            raise FileNotFoundError("data/raw not found — run `make seed`")


        corpus = find_corpus()
        pdfs = sorted(corpus.glob("*.pdf"))
        print(f"{len(pdfs)} PDFs in {corpus}")
        for path in pdfs:
            print(" -", path.name)
        


## Extractors


In [ ]:
        def extract_pymupdf(path):
            import pymupdf

            with pymupdf.open(path) as doc:
                return [page.get_text("text") for page in doc]


        def extract_pypdf(path):
            from pypdf import PdfReader

            return [(page.extract_text() or "") for page in PdfReader(str(path)).pages]
        


## Measure


In [ ]:
        import time

        import pandas as pd

        rows = []
        for path in pdfs:
            for name, fn in (("pymupdf", extract_pymupdf), ("pypdf", extract_pypdf)):
                start = time.perf_counter()
                pages = fn(path)
                seconds = time.perf_counter() - start
                rows.append(
                    {
                        "pdf": path.name[:34],
                        "extractor": name,
                        "pages": len(pages),
                        "chars": sum(len(text) for text in pages),
                        "seconds": round(seconds, 2),
                    }
                )

        df = pd.DataFrame(rows)
        agg = df.groupby("extractor").agg(
            pdfs=("pdf", "count"),
            pages=("pages", "sum"),
            chars=("chars", "sum"),
            seconds=("seconds", "sum"),
        )
        agg["chars_per_page"] = (agg["chars"] / agg["pages"]).round(0)
        agg[["pdfs", "pages", "chars_per_page", "seconds"]]
        


        ## Decision

        - **Chosen:** `pymupdf`.
        - **Why:** faster, and better reading order on these documents. Both
          return a list indexed by page, so both preserve `#page=N`.
        - **Trade-off:** neither reconstructs complex tables; PyMuPDF can
          reorder columns. Acceptable for v1 — page text is stored alongside
          chunks, so the model can read the surrounding rows. A table-aware or
          VLM pass is a backlog item.
        - **ADR:** [ADR-0005](../../docs/adr/0005-page-aware-ingestion.md)

        The numbers above are the evidence; if they ever disagree with this
        decision, change the decision.
        
